# **Criteo Uplift Dataset: Data Understanding**

## **Objective**

This notebook examines the structure, variables, and data quality of the
Criteo Uplift Prediction Dataset before conducting incrementality analysis.

The analysis will:

1. Review the dataset dimensions, columns, and data types.
2. Examine the treatment, exposure, visit, and conversion variables.
3. Check missing values and binary-variable validity.
4. Validate key relationships between the experiment outcomes.
5. Document important limitations of the anonymized feature variables.

The dataset contains:

- 12 anonymized user features (`f0`–`f11`)
- Randomized treatment assignment
- Actual advertising exposure
- Website visit outcome
- Conversion outcome

In [35]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


## **Data Loading**

The Criteo Uplift Prediction Dataset is loaded from the compressed CSV file.

Each row represents one experimental observation.

In [36]:
df = pd.read_csv("../data/raw/criteo-uplift-v2.1.csv.gz")

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

display(df.head())


Dataset loaded successfully!
Rows: 13,979,592
Columns: 16


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [37]:
# Basic dataset overview 
column_summary = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "unique_values": df.nunique().values
})

display(column_summary)


,column,data_type,non_null_count,missing_count,unique_values
0,f0,float64,13979592,0,2181959
1,f1,float64,13979592,0,60
2,f2,float64,13979592,0,2051900
3,f3,float64,13979592,0,552
4,f4,float64,13979592,0,260
5,f5,float64,13979592,0,132
6,f6,float64,13979592,0,1645
7,f7,float64,13979592,0,622143
8,f8,float64,13979592,0,3743
9,f9,float64,13979592,0,1594


In [38]:
# Data types and missing_value overview 

df.info(show_counts = True)


<class 'pandas.DataFrame'>
RangeIndex: 13979592 entries, 0 to 13979591
Data columns (total 16 columns):
 #   Column      Non-Null Count     Dtype  
---  ------      --------------     -----  
 0   f0          13979592 non-null  float64
 1   f1          13979592 non-null  float64
 2   f2          13979592 non-null  float64
 3   f3          13979592 non-null  float64
 4   f4          13979592 non-null  float64
 5   f5          13979592 non-null  float64
 6   f6          13979592 non-null  float64
 7   f7          13979592 non-null  float64
 8   f8          13979592 non-null  float64
 9   f9          13979592 non-null  float64
 10  f10         13979592 non-null  float64
 11  f11         13979592 non-null  float64
 12  treatment   13979592 non-null  int64  
 13  conversion  13979592 non-null  int64  
 14  visit       13979592 non-null  int64  
 15  exposure    13979592 non-null  int64  
dtypes: float64(12), int64(4)
memory usage: 1.7 GB


## **Experiment Variables**

The dataset contains four binary experiment-related variables:

| Variable | Definition |
|---|---|
| `treatment` | Whether the observation was assigned to the advertising treatment group |
| `exposure` | Whether the observation was actually exposed to the advertisement |
| `visit` | Whether the user visited the advertiser's website |
| `conversion` | Whether the user completed the target conversion |

The primary analysis follows an intention-to-treat approach using
`treatment` as the experimental assignment variable.

`exposure` is a post-assignment delivery variable and is not used to redefine
the treatment group.

In [39]:
experiment_columns = [
    "treatment",
    "exposure",
    "visit",
    "conversion"
]

experiment_variable_summary = pd.DataFrame({
    "variable": experiment_columns,
    "count_0": [
        (df[column] == 0).sum()
        for column in experiment_columns
    ],
    "count_1": [
        (df[column] == 1).sum()
        for column in experiment_columns
    ],
    "rate_1": [
        df[column].mean()
        for column in experiment_columns
    ]
})

display(experiment_variable_summary)


,variable,count_0,count_1,rate_1
0,treatment,2096937,11882655,0.850000
1,exposure,13551380,428212,0.030631
2,visit,13322663,656929,0.046992
3,conversion,13938818,40774,0.002917


In [40]:
experiment_variable_summary_display = (
    experiment_variable_summary.copy()
)

experiment_variable_summary_display["rate_1"] = (
    experiment_variable_summary_display["rate_1"]
    .map(lambda value: f"{value:.2%}")
)

display(experiment_variable_summary_display)

,variable,count_0,count_1,rate_1
0,treatment,2096937,11882655,85.00%
1,exposure,13551380,428212,3.06%
2,visit,13322663,656929,4.70%
3,conversion,13938818,40774,0.29%


### **Initial Observations**

- Approximately 85% of observations were assigned to the treatment group,
  while 15% were assigned to the control group.
- Actual advertising exposure occurred for only a small share of the total
  sample.
- Website visits were substantially more common than final conversions.
- Conversion is a relatively rare outcome, with an overall rate below 1%.
- Because the experiment groups have different sample sizes, outcome rates
  should be compared instead of raw counts.

## **Data Quality Checks**

This section evaluates missing values, binary-variable validity, and logical
relationships among the experiment variables.

In [41]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
})

missing_summary = (
    missing_summary
    .reset_index()
    .rename(columns={
        "index": "column"
    })
)

display(missing_summary)


,column,missing_count
0,f0,0
1,f1,0
2,f2,0
3,f3,0
4,f4,0
5,f5,0
6,f6,0
7,f7,0
8,f8,0
9,f9,0


In [42]:
binary_columns = [
    "treatment",
    "exposure",
    "visit",
    "conversion"
]

binary_validation = pd.DataFrame({
    "column": binary_columns,
    "valid_binary": [
        df[column].isin([0, 1]).all()
        for column in binary_columns
    ],
    "invalid_values": [
        (~df[column].isin([0, 1])).sum()
        for column in binary_columns
    ]
})

display(binary_validation)

,column,valid_binary,invalid_values
0,treatment,True,0
1,exposure,True,0
2,visit,True,0
3,conversion,True,0


In [43]:
quality_checks = {
    "Control users marked as exposed": (
        (df["treatment"] == 0)
        & (df["exposure"] == 1)
    ).sum(),

    "Conversions without a visit": (
        (df["conversion"] == 1)
        & (df["visit"] == 0)
    ).sum(),

    "Exposed users outside treatment": (
        (df["exposure"] == 1)
        & (df["treatment"] != 1)
    ).sum(),

    "Invalid treatment values": (
        ~df["treatment"].isin([0, 1])
    ).sum(),

    "Invalid exposure values": (
        ~df["exposure"].isin([0, 1])
    ).sum(),

    "Invalid visit values": (
        ~df["visit"].isin([0, 1])
    ).sum(),

    "Invalid conversion values": (
        ~df["conversion"].isin([0, 1])
    ).sum()
}

quality_check_summary = (
    pd.Series(
        quality_checks,
        name="issue_count"
    )
    .rename_axis("quality_check")
    .reset_index()
)

quality_check_summary["passed"] = (
    quality_check_summary["issue_count"] == 0
)

display(quality_check_summary)

,quality_check,issue_count,passed
0,Control users marked as exposed,0,True
1,Conversions without a visit,0,True
2,Exposed users outside treatment,0,True
3,Invalid treatment values,0,True
4,Invalid exposure values,0,True
5,Invalid visit values,0,True
6,Invalid conversion values,0,True


In [44]:
duplicate_rows = df.duplicated().sum()

print(f"Fully duplicated rows: {duplicate_rows:,}")

Fully duplicated rows: 1,259,545


### **Duplicate-Row Interpretation**

The dataset does not include a unique user or observation identifier.
Therefore, rows with identical values cannot automatically be classified as
erroneous duplicates.

Multiple users may legitimately share the same anonymized features and
experiment outcomes. Fully identical rows are documented but are not removed.

In [45]:
data_quality_summary = pd.DataFrame({
    "check": [
        "Missing values",
        "Invalid binary values",
        "Control users marked as exposed",
        "Conversions without a visit"
    ],
    "issue_count": [
        int(df.isna().sum().sum()),
        int(
            sum(
                (~df[column].isin([0, 1])).sum()
                for column in binary_columns
            )
        ),
        int(
            (
                (df["treatment"] == 0)
                & (df["exposure"] == 1)
            ).sum()
        ),
        int(
            (
                (df["conversion"] == 1)
                & (df["visit"] == 0)
            ).sum()
        )
    ]
})

data_quality_summary["passed"] = (
    data_quality_summary["issue_count"] == 0
)

display(data_quality_summary)

,check,issue_count,passed
0,Missing values,0,True
1,Invalid binary values,0,True
2,Control users marked as exposed,0,True
3,Conversions without a visit,0,True


## **Data Understanding Findings**

- The dataset contains **13,979,592 observations** and **16 columns**.

- Twelve columns (`f0`–`f11`) represent anonymized user-level features.
  Their actual demographic, behavioral, or commercial meanings are not
  provided.

- The dataset contains four experiment-related binary variables:
  `treatment`, `exposure`, `visit`, and `conversion`.

- Approximately **85%** of observations were assigned to the treatment group,
  while approximately **15%** were assigned to the control group.

- A total of **428,212 observations** were marked as actually exposed to the
  advertisement.

- The dataset contains **656,929 website visits** and **40,774 conversions**.

- Website visits occurred more frequently than conversions, indicating a
  substantial drop between upper-funnel engagement and the final conversion
  outcome.

- Conversion is a relatively rare outcome, with an overall rate below 1%.
  The large sample size is therefore important for estimating treatment
  effects precisely.

- No missing values were identified, and all experiment-related variables
  contained only valid binary values.

- No control observations were marked as exposed, which is consistent with
  the expected experiment structure.

- No conversions occurred without a recorded visit, supporting the logical
  consistency of the outcome variables.

- Because the treatment and control groups have different sample sizes,
  experiment performance should be evaluated using rates, treatment effects,
  confidence intervals, and statistical tests rather than raw outcome counts.

## **Experiment Design Note**

The primary analysis follows an **intention-to-treat (ITT)** approach.

Users are analyzed according to their randomized treatment assignment:

- `treatment = 0`: assigned to the control group
- `treatment = 1`: assigned to the advertising treatment group

The `exposure` variable indicates whether a treatment-assigned observation was
actually exposed to the advertisement. It is a post-assignment delivery
variable rather than the original randomized assignment.

The main causal comparison therefore uses the treatment and control groups,
regardless of whether every treatment-assigned observation was actually
exposed.

Restricting the treatment group to exposed observations and comparing them
directly with the control group could introduce selection bias because actual
exposure may depend on post-assignment behavior or advertising-delivery
conditions.

As a result:

- `treatment` is used to estimate the primary incremental effect.
- `exposure` is used as a campaign-delivery diagnostic.
- The estimated treatment effect represents the impact of assignment to the
  advertising strategy, not necessarily the effect of viewing one individual
  advertisement.

## **Data and Design Limitations**

- **Anonymized features:**  
  The meanings of `f0` through `f11` are not provided. Statistical patterns
  involving these variables cannot be translated directly into demographic,
  behavioral, or customer-level interpretations.

- **No unique observation identifier:**  
  The dataset does not include a user ID or observation ID. Rows with identical
  values cannot be confirmed as duplicate records and should not be removed
  automatically.

- **Unequal experiment allocation:**  
  The treatment group is substantially larger than the control group. This
  does not invalidate rate-based treatment-effect estimation, but raw outcome
  counts should not be compared directly.

- **Incomplete treatment delivery:**  
  Only a subset of treatment-assigned observations were actually exposed to
  the advertisement. The intention-to-treat estimate may therefore be diluted
  relative to the effect among users who actually received the advertisement.

- **Rare conversion outcome:**  
  Conversion occurs for only a small proportion of observations. Segment-level
  analysis must consider sample size, confidence intervals, and statistical
  uncertainty.

- **No campaign cost or revenue information:**  
  The dataset supports estimation of incremental visits and conversions but
  does not contain the financial information needed to calculate ROI,
  profitability, or optimal advertising spend.

- **No time variable:**  
  The dataset does not provide experiment dates or observation timestamps.
  Time trends, seasonality, experiment duration, and novelty effects cannot be
  evaluated.

- **No external validation sample:**  
  Findings based on this experiment should be validated using a future
  randomized experiment or a separate holdout sample before operational use. 

## **Conclusion**

The dataset is suitable for treatment-control incrementality analysis. The
experiment variables are complete, binary, and logically consistent, and the
large sample provides sufficient observations for analyzing rare conversion
outcomes.

The main experiment analysis will use treatment assignment to estimate
intention-to-treat effects on website visits and conversions. The anonymized
features will later be used for exploratory segment analysis, with appropriate
caution regarding interpretation and overlapping customer groups.